# Bug Report Validity Detection

## Data Preprocessing

In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import re
import html
from bs4 import BeautifulSoup

In [2]:
data_path = "data"

train_data = pd.read_csv(os.path.join(data_path, "issues_train.csv"))
test_data = pd.read_csv(os.path.join(data_path, "issues_test.csv"))

print("Train data shape:", train_data.shape)
print("Test data shape:", test_data.shape)

Train data shape: (1500, 5)
Test data shape: (1500, 5)


In [3]:
# Label distribution
print("Train label counts:\n", train_data['label'].value_counts(dropna=False))
print("Nulls per column:\n", train_data.isnull().sum())
# Duplicates
print("Duplicate rows in train:", train_data.duplicated().sum())

Train label counts:
 label
bug         500
feature     500
question    500
Name: count, dtype: int64
Nulls per column:
 repo          0
created_at    0
label         0
title         0
body          0
dtype: int64
Duplicate rows in train: 0


In [ ]:
# Keep as DataFrames so we can add text-based columns/flags later
trainX = train_data.drop("label", axis=1).copy()
trainY = train_data["label"]  # keep as a Series for now

testX = test_data.drop("label", axis=1).copy()
testY = test_data["label"]

print("Processed trainX shape:", trainX.shape)
print("Processed testX shape:", testX.shape)

Processed trainX shape: (1500, 4)
Processed testX shape: (1500, 4)


In [6]:
URL_RE = re.compile(r'https?://\S+|www\.\S+')
EMAIL_RE = re.compile(r'\S+@\S+')
HEX_RE = re.compile(r'0x[a-fA-F0-9]+')
STACKTRACE_RE = re.compile(r'(Traceback|Exception|Error|at\s+\w+\()')  # basic heuristic
CODEBLOCK_RE = re.compile(r'```.*?```', re.DOTALL)

def clean_text(text):
    if pd.isna(text):
        return ""
    # unescape HTML entities, remove HTML tags
    text = html.unescape(text)
    text = BeautifulSoup(text, "html.parser").get_text(separator=" ")
    # mask URLs and emails
    text = URL_RE.sub(" <URL> ", text)
    text = EMAIL_RE.sub(" <EMAIL> ", text)
    # remove code fences (but keep a flag separately)
    text = CODEBLOCK_RE.sub(" <CODE> ", text)
    # remove hex dumps, long hex sequences
    text = HEX_RE.sub(" <HEX> ", text)
    # collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# flags
trainX['has_stacktrace'] = trainX['text'].str.contains(STACKTRACE_RE)
testX['has_stacktrace'] = testX['text'].str.contains(STACKTRACE_RE)

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices